# L63 Proof Replay

Replays the bounded workload certification proof on fixed repo state.

Proof focuses on:
- Four ordered certification phases
- Five governance rules
- Per-domain certification verdicts matching routing table
- Zero-action domains fail-closed to not-yet-certified
- Certified + not-yet-certified = total domain count

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()
contract = yaml.safe_load(
    (repo_root / 'registry' / 'workload_certification_v1.yaml').read_text(encoding='utf-8')
)
routing = yaml.safe_load(
    (repo_root / 'registry' / 'executor_routing_v2.yaml').read_text(encoding='utf-8')
)

# Phase validation
assert len(contract['certification_phases']) == 4
assert [p['order'] for p in contract['certification_phases']] == [1, 2, 3, 4]
for phase in contract['certification_phases']:
    assert all(k in phase for k in ('id', 'description', 'inputs', 'outputs', 'failure_mode'))

# Governance rules
assert set(contract['governance_rules'].keys()) == {
    'fail_closed', 'audit_completeness', 'bounded_claims',
    'determinism', 'wave_alignment',
}

# Domain certification alignment
domains = routing['canonical_executor_domains']
routes = routing.get('exact_action_routes', {})
domain_counts = {d: 0 for d in domains}
for action, domain in routes.items():
    if domain in domain_counts:
        domain_counts[domain] += 1

cert_status = contract['domain_certification_status']
certified = 0
not_yet = 0
total_actions = 0
for d in domains:
    entry = cert_status[d]
    assert entry['routed_action_count'] == domain_counts[d]
    total_actions += entry['routed_action_count']
    if domain_counts[d] == 0:
        assert entry['certification_verdict'] == 'not-yet-certified'
        not_yet += 1
    else:
        assert entry['certification_verdict'] == 'certified'
        certified += 1

kpis = contract['kpis']
assert kpis['total_executor_domains'] == len(domains)
assert kpis['domains_with_routed_actions'] == certified
assert kpis['domains_without_routed_actions'] == not_yet
assert kpis['total_routed_actions'] == total_actions
assert certified + not_yet == len(domains)